# Heady™ Embedding Fine-Tuning — Domain-Specialized Vector Memory
© 2026 HeadySystems Inc. — Eric Haywood

Fine-tunes a **Nomic Embed v1.5** model on the Heady codebase to produce
domain-specialized embeddings for CSL gates, bee routing, and vector search.

**Runtime:** A100 GPU (Colab Pro+) — estimated 2–4 hours for full fine-tune

**Output:** A GGUF/ONNX model artifact + pgvector-ready embedding function

In [ ]:
# ═══ Cell 1: Install Dependencies ══════════════════════════════════════════════════
!pip install -q \
    sentence-transformers==3.4.1 \
    datasets \
    accelerate \
    peft \
    wandb \
    torch \
    transformers \
    huggingface_hub \
    onnxruntime-gpu \
    numpy \
    pynvml

print('\u2713 Dependencies installed')

In [ ]:
# ═══ Cell 2: Authenticate & Configure ══════════════════════════════════════════
import os
from google.colab import userdata

# API key for Heady gateway auth (set in Colab Secrets)
os.environ['HEADY_API_KEY'] = userdata.get('HEADY_API_KEY')

# Gateway URL
HEADY_GATEWAY_URL = 'https://colab-gateway.headysystems.com'
POOL = 'warm'  # Fine-tuning runs on the warm pool

# Model config
BASE_MODEL = 'nomic-ai/nomic-embed-text-v1.5'
OUTPUT_NAME = 'heady-embed-v1'
EMBEDDING_DIM = 768
MAX_SEQ_LENGTH = 512

# φ-scaled training params (golden ratio derived)
PHI = 1.618033988749895
LEARNING_RATE = 2e-5 * PHI       # ~3.24e-5
WARMUP_RATIO = 1 / PHI           # 0.618
BATCH_SIZE = 13                  # fib(7)
EPOCHS = 3                       # fib(4)
EVAL_STEPS = 89                  # fib(11)

print(f'Base model: {BASE_MODEL}')
print(f'Output name: {OUTPUT_NAME}')
print(f'LR: {LEARNING_RATE:.6f}, Batch: {BATCH_SIZE}, Epochs: {EPOCHS}')

In [ ]:
# ═══ Cell 3: GPU Verification ════════════════════════════════════════════════
import torch
import pynvml

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
info = pynvml.nvmlDeviceGetMemoryInfo(handle)
gpu_name = pynvml.nvmlDeviceGetName(handle)

print(f'GPU: {gpu_name}')
print(f'VRAM: {info.total / 1024**3:.1f} GB')
print(f'CUDA: {torch.cuda.is_available()}')
print(f'PyTorch: {torch.__version__}')

assert torch.cuda.is_available(), 'CUDA not available — switch to GPU runtime'

In [ ]:
# ═══ Cell 4: Clone Heady Repo & Extract Training Corpus ═════════════════════
import glob
import json
import hashlib
from pathlib import Path

# Clone the codebase (shallow for speed)
!git clone --depth 1 https://github.com/HeadyMe/Heady-Testing.git /tmp/heady 2>/dev/null || echo 'Repo already cloned'

# File extensions to include for training corpus
CODE_EXTS = {'.js', '.ts', '.py', '.md', '.yaml', '.yml', '.json', '.html', '.css'}
SKIP_DIRS = {'node_modules', '.git', 'dist', 'build', '.turbo', '__pycache__'}

def extract_corpus(root_dir, max_chunk_chars=2048):
    """Extract code chunks with metadata for contrastive learning."""
    corpus = []
    for path in Path(root_dir).rglob('*'):
        if not path.is_file():
            continue
        if path.suffix not in CODE_EXTS:
            continue
        if any(skip in path.parts for skip in SKIP_DIRS):
            continue
        try:
            content = path.read_text(errors='ignore')
            if len(content) < 50:  # Skip tiny files
                continue
            
            rel_path = str(path.relative_to(root_dir))
            
            # Split into chunks for training
            for i in range(0, len(content), max_chunk_chars):
                chunk = content[i:i + max_chunk_chars]
                corpus.append({
                    'text': f'search_document: {rel_path}\n{chunk}',
                    'path': rel_path,
                    'ext': path.suffix,
                    'chunk_idx': i // max_chunk_chars,
                    'hash': hashlib.md5(chunk.encode()).hexdigest()[:8],
                })
        except Exception:
            continue
    return corpus

corpus = extract_corpus('/tmp/heady')
print(f'\u2713 Extracted {len(corpus)} training chunks from codebase')

# Show distribution by file type
from collections import Counter
ext_counts = Counter(c['ext'] for c in corpus)
for ext, count in ext_counts.most_common():
    print(f'  {ext}: {count} chunks')

In [ ]:
# ═══ Cell 5: Build Contrastive Training Pairs ══════════════════════════════
import random
from datasets import Dataset

def build_training_pairs(corpus, num_negatives=5):
    """Build (anchor, positive, negative) triplets for contrastive learning.
    
    Positive: another chunk from the same file (semantic neighbor)
    Negative: a chunk from a different directory (semantic distant)
    """
    # Group by file path
    by_path = {}
    for item in corpus:
        by_path.setdefault(item['path'], []).append(item)
    
    # Group by directory (for hard negatives)
    by_dir = {}
    for item in corpus:
        dir_name = '/'.join(item['path'].split('/')[:-1]) or 'root'
        by_dir.setdefault(dir_name, []).append(item)
    
    pairs = []
    for path, chunks in by_path.items():
        if len(chunks) < 2:
            continue
        
        dir_name = '/'.join(path.split('/')[:-1]) or 'root'
        other_dirs = [d for d in by_dir if d != dir_name]
        
        for i, anchor in enumerate(chunks):
            # Positive: adjacent chunk from same file
            pos_idx = (i + 1) % len(chunks)
            positive = chunks[pos_idx]
            
            # Negatives: random chunks from different directories
            negatives = []
            if other_dirs:
                for _ in range(num_negatives):
                    neg_dir = random.choice(other_dirs)
                    neg_chunk = random.choice(by_dir[neg_dir])
                    negatives.append(neg_chunk['text'])
            
            pairs.append({
                'anchor': anchor['text'],
                'positive': positive['text'],
                'negative': negatives[0] if negatives else '',
            })
    
    random.shuffle(pairs)
    return pairs

pairs = build_training_pairs(corpus)
print(f'\u2713 Built {len(pairs)} training pairs')

# Split into train/eval (using φ ratio)
split_idx = int(len(pairs) * (1 / PHI))  # 61.8% train, 38.2% eval
train_pairs = pairs[:split_idx]
eval_pairs = pairs[split_idx:]

train_ds = Dataset.from_list(train_pairs)
eval_ds = Dataset.from_list(eval_pairs)

print(f'  Train: {len(train_ds)} | Eval: {len(eval_ds)}')

In [ ]:
# ═══ Cell 6: Load Base Model & Fine-Tune ═════════════════════════════════════
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)

# Load base model
model = SentenceTransformer(BASE_MODEL, trust_remote_code=True)
model.max_seq_length = MAX_SEQ_LENGTH
print(f'\u2713 Loaded {BASE_MODEL}')
print(f'  Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'  Embedding dim: {model.get_sentence_embedding_dimension()}')

# Contrastive loss (TripletLoss with φ-scaled margin)
loss = losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=1 / PHI,  # 0.618 margin
)

# Training arguments
args = SentenceTransformerTrainingArguments(
    output_dir=f'/tmp/{OUTPUT_NAME}',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=EVAL_STEPS * 2,  # fib(11)*2 = 178
    logging_steps=21,           # fib(8)
    fp16=True,
    dataloader_num_workers=4,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

# Train
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    loss=loss,
)

print('\n\u2550' * 60)
print('Starting fine-tuning...')
print('\u2550' * 60)
trainer.train()
print('\n\u2713 Fine-tuning complete!')

In [ ]:
# ═══ Cell 7: Evaluate & Compare ═════════════════════════════════════════════
import numpy as np

# Load fine-tuned model
ft_model = SentenceTransformer(f'/tmp/{OUTPUT_NAME}')

# Load base model for comparison
base_model = SentenceTransformer(BASE_MODEL, trust_remote_code=True)

# Heady-specific test queries
test_queries = [
    'search_query: CSL cosine gate routing for bee swarm task decomposition',
    'search_query: phi-math sacred geometry Fibonacci scaled constants',
    'search_query: auth session server httpOnly cookie creation Firebase Admin',
    'search_query: pgvector embedding storage HNSW index creation',
    'search_query: HeadyBuddy conversational agent voice relay interface',
    'search_query: Cloudflare Workers edge inference latent OS routing',
    'search_query: onboarding flow user profile Drupal JSON API',
    'search_query: colab gateway warm pool fine-tuning batch processing',
]

# Get test corpus embeddings
test_texts = [c['text'] for c in random.sample(corpus, min(500, len(corpus)))]

base_corpus_emb = base_model.encode(test_texts, show_progress_bar=True, batch_size=32)
ft_corpus_emb = ft_model.encode(test_texts, show_progress_bar=True, batch_size=32)

base_query_emb = base_model.encode(test_queries, batch_size=8)
ft_query_emb = ft_model.encode(test_queries, batch_size=8)

# Compare top-5 retrieval quality
from sentence_transformers.util import cos_sim

print('\n' + '\u2550' * 70)
print('RETRIEVAL COMPARISON: Base vs Fine-Tuned')
print('\u2550' * 70)

for i, query in enumerate(test_queries):
    base_scores = cos_sim(base_query_emb[i:i+1], base_corpus_emb)[0]
    ft_scores = cos_sim(ft_query_emb[i:i+1], ft_corpus_emb)[0]
    
    base_top = float(base_scores.max())
    ft_top = float(ft_scores.max())
    
    improvement = ((ft_top - base_top) / base_top * 100) if base_top > 0 else 0
    arrow = '\u2191' if improvement > 0 else '\u2193'
    
    short_q = query.replace('search_query: ', '')[:55]
    print(f'  {short_q:<55} base={base_top:.3f} ft={ft_top:.3f} {arrow}{abs(improvement):.1f}%')

# Overall stats
base_all = cos_sim(base_query_emb, base_corpus_emb)
ft_all = cos_sim(ft_query_emb, ft_corpus_emb)
print(f'\n  Mean top-1 sim (base):      {float(base_all.max(dim=1).values.mean()):.4f}')
print(f'  Mean top-1 sim (fine-tuned): {float(ft_all.max(dim=1).values.mean()):.4f}')

In [ ]:
# ═══ Cell 8: Export & Save Model ═══════════════════════════════════════════
import shutil

# Save final model
final_path = f'/tmp/{OUTPUT_NAME}-final'
ft_model.save(final_path)
print(f'\u2713 Model saved to {final_path}')

# Create zip for download / deploy
zip_path = shutil.make_archive(
    f'/content/{OUTPUT_NAME}',
    'zip',
    final_path
)
print(f'\u2713 Model archived: {zip_path}')
print(f'  Size: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB')

# Download link (Colab)
from google.colab import files
files.download(zip_path)
print('\n\u2713 Download triggered — save the model locally')

In [ ]:
# ═══ Cell 9: Register with Gateway ══════════════════════════════════════════
import requests

api_key = os.environ.get('HEADY_API_KEY', '')

# Notify gateway that warm pool now has the fine-tuned model
try:
    resp = requests.post(
        f'{HEADY_GATEWAY_URL}/runtime/warm/register',
        headers={'x-heady-api-key': api_key},
        json={
            'capabilities': ['embedding', 'fine-tune', 'batch-process', 'hnsw-build'],
            'models': [OUTPUT_NAME, BASE_MODEL],
            'gpuType': gpu_name if 'gpu_name' in dir() else 'A100',
            'vram': info.total // (1024**3) if 'info' in dir() else 80,
        },
        timeout=10,
    )
    print(f'Gateway registration: {resp.status_code} {resp.json()}')
except Exception as e:
    print(f'Gateway not reachable (normal for first run): {e}')
    print('Manual registration: POST /runtime/warm/register with x-heady-api-key header')